# CMLC/GMLC Sentinel-1 demo

In [ ]:
"""
File Name: cmlc_gmlc_sentinel_1.ipynb

Author:
    Francesco De Zan
    delta phi remote sensing GmbH

Date: June 2025

Description:
------------
This script performs the generation of CMLC and GMLC for a stack of CSLCs from the SENTINEL-1 sensor. 
Options and configuration parameters about some processing steps can be set
by the user.

Usage:
------
To use this script, define the following parameters:
    
    - cslc_file_name
    - cmlc_file_name
    - gmlc_file_name

Optionally, set the processing parameters:
    - w_rg: the range extension of the multi-looking window
    - w_az: the azimuth extension of the multi-looking window
    - subsampling: the sub-sampling factor for the GMLC compared to a 10 m x 10 m grid
    - mini_size: the number of acquisitions contributing to the generation of the "past reference"
    - memory_param: the autoregression parameter for the reference update (0 => totally forgetting the past)
    - memory_param_past = 1.0: the autoregression parameter for the past reference update (0 => totally forgetting the past)
    - memory_param_ma: the moving average parameter for the past reference update (0 => freezing past reference)
    - coherence_threshold: the threshold to calibrate current reference with a past reference (below that, skip calibration)

"""

In [ ]:
## Input cslc product path

base_dir = "/media/data/SAR/L2/Naples/S1/044/"
#base_dir = "/media/data/SAR/L2/Naples/S1/124/"

channel_dir = "SENTINEL1_044_20230611_20240723_VV/"
#channel_dir = "SENTINEL1_124_20230617_20240729_VV/"

## output CMLC path
cmlc_file_name = base_dir + channel_dir + "cmlcs_test"

## output GMLC path
gmlc_file_name = cmlc_file_name + ".gmlc"

In [ ]:
# selecting window sizes (rg, az) and mini_size (number of acquisitions after which the long-term calibration kicks-in)
w_rg = 20
w_az = 5

multilook = {"azimuth_time": w_az, "slant_range_time": w_rg}

# subsampling for geocoding: default is sampled @ 10m x 10m, we want 50m x 50m
subsampling = 5

#mini_size = 100 # this setting disables the long-term calibration, since 100 > N
mini_size = 10

memory_param = 0.98 # autoregression parameter for the reference update (0 => totally forgetting the past)

memory_param_past = 1.0 # autoregression parameter for the past reference update (0 => totally forgetting the past)
memory_param_ma = 0.0 # moving average parameter for the past reference update (0 => freezing past reference)

coherence_threshold = 0.1 # threshold to calibrate current reference with a past reference (below that, skip calibration)

In [ ]:
import numpy as np
import datatree
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil
import dask
import rioxarray

In [ ]:
import rosel.io
import rosel.computation
import rosel.shiroma

In [ ]:
type, n_jobs, job_memory = ["LOCAL", 16, None]
client = rosel.computation.create_client(type, n_jobs, job_memory)

In [ ]:
## Opening clsc product
cslc_file_name = base_dir + channel_dir
ds = datatree.open_datatree(cslc_file_name, engine='zarr', chunks={})

In [ ]:
# finding the names of all the bursts
bursts = tuple(item for item in ds['bursts'].groups if "coreg" in item)

In [ ]:
def make_interf(A, B):
    return np.conj(A).dot(B)

def make_coh(A, B):
    return make_interf(A,B) / np.sqrt(make_interf(A,A) * make_interf(B,B))

In [ ]:
# CMLC phase estimation with apply_ufunc

def cmlc_phase_estimation(cstack):
    step = 0
    current_reference = cstack[0].ravel()
    current_reference = np.zeros_like(current_reference)
    current_reference[:] = cstack[0].ravel()
    past_reference = np.zeros_like(current_reference)
    interf = np.zeros(N, dtype=np.complex64)
    short_term_coherence = np.zeros(N)*np.nan
    long_term_coherence = np.zeros(N)*np.nan
    for n in range(0,N,1):
        step = step + 1
        slc = cstack[n].ravel()  # getting the current slc
        interf[n] = make_coh(current_reference,slc)
        short_term_coherence[n] = abs(interf[n])
        interf[n] = interf[n] / abs(interf[n]) # normalize interferogram

        # update the current reference
        if ~np.isnan(interf[n]):
            current_reference[:] = current_reference*memory_param + slc*np.conj(interf[n])

        # generate (a) the past reference
        if step == mini_size :
            past_reference[:] = current_reference
        
        # long-term phase calibration of current_reference
        if step > mini_size :
            corr_phase = make_coh(past_reference,current_reference)
            if np.abs(corr_phase) > coherence_threshold :
                current_reference[:] = current_reference * (np.conj(corr_phase)/np.abs(corr_phase))
            long_term_coherence[n] = np.abs(make_coh(past_reference,slc))
            past_reference[:] = past_reference[:] * memory_param_past + current_reference[:] * memory_param_ma

    return interf, short_term_coherence, long_term_coherence

In [ ]:
# initializing the DataTree
cmlc = datatree.DataTree()

for n in tqdm(range(0,len(bursts),1)):

    cstack = ds[bursts[n]].cstack

    N = len(cstack)
    n_lines, n_samples = cstack[0].shape
    n_rg = np.floor_divide(n_samples, w_rg)
    n_az = np.floor_divide(n_lines, w_az)

    cstack_temp = cstack.coarsen(azimuth_time=w_az, slant_range_time=w_rg, boundary='trim').construct(
        azimuth_time=("az_coarse", "az_window"), slant_range_time=("rg_coarse", "rg_window"), keep_attrs=True)
    cstack_temp = cstack_temp.chunk(dict(acquisition_date=N)) # no chunks allows in core dimensions!

    interf, short_term_coherence, long_term_coherence = xr.apply_ufunc(
        cmlc_phase_estimation, 
        cstack_temp,
        input_core_dims=[["acquisition_date", "az_window", "rg_window"]],
        output_core_dims=[["acquisition_date"], ["acquisition_date"], ["acquisition_date"]],
        output_dtypes=[cstack_temp.dtype, np.float64, np.float64],
        vectorize=True,
        dask="parallelized"
    )

    #computing
    interf, short_term_coherence, long_term_coherence = dask.compute(interf, short_term_coherence, long_term_coherence)

    # transposing
    interf = interf.transpose('acquisition_date', 'az_coarse', 'rg_coarse')
    short_term_coherence = short_term_coherence.transpose('acquisition_date', 'az_coarse', 'rg_coarse')
    long_term_coherence = long_term_coherence.transpose('acquisition_date', 'az_coarse', 'rg_coarse')

    # collecting and formatting variables
    attrs = cstack.attrs
    attrs['multilook'] = multilook

    cmlc_ds = xr.Dataset(
        data_vars=dict(
            phase=(["acquisition_date", "azimuth_time", "slant_range_time"], interf.data),
            long_term_coherence=(["acquisition_date", "azimuth_time", "slant_range_time"], long_term_coherence.data),
            short_term_coherence=(["acquisition_date", "azimuth_time", "slant_range_time"], short_term_coherence.data),
        ),

        coords=dict(
            azimuth_time=("azimuth_time", cstack[0].azimuth_time.coarsen(azimuth_time=w_az, boundary='trim').mean().data),
            slant_range_time=("slant_range_time", cstack[0].slant_range_time.coarsen(slant_range_time=w_rg, boundary='trim').mean().data),
            acquisition_date=("acquisition_date", cstack.acquisition_date.data),
        ),

        #attrs=dict(description="Co-registered Multi-look Complex"),
        attrs=attrs,
    )

    # saving in DataTree
    cmlc[bursts[n]] = datatree.DataTree(cmlc_ds)

In [ ]:
#saving
cmlc.to_zarr(cmlc_file_name)

In [ ]:
# loading CMLC
ds_cmlc = datatree.open_datatree(cmlc_file_name, engine='zarr', chunks={})

In [ ]:
def geocode_3D(cstack, data_path_shiroma, cmlc_input, gmlc_output, f, x, y):

    for n in tqdm(range(0,N)):
    #for n in tqdm(range(0,2)):
        # save CMLC (temp)
        cmlc_temp_file_name = f"{base_dir}/cmlc_temp"
        try:
            shutil.rmtree(cmlc_temp_file_name)
        except OSError as e:
            print("Error: %s - %s." % (e.filename, e.strerror))
    
        cmlc = cmlc_input[n] # ds_cmlc.phase[n]
        #pad_az = (cstack[0].azimuth_time.coarsen(azimuth_time=w_az, boundary='pad')).mean().size - (cstack[0].azimuth_time.coarsen(azimuth_time=w_az, boundary='trim')).mean().size
        #pad_rg = (cstack[0].slant_range_time.coarsen(slant_range_time=w_rg, boundary='pad')).mean().size - (cstack[0].slant_range_time.coarsen(slant_range_time=w_rg, boundary='trim')).mean().size
        cmlc = np.pad(cmlc.data, pad_width=((0,1),(0,1))) # these should be computed! padding is assumed!
        cmlc = xr.DataArray(cmlc, dims=['azimuth_time', 'slant_range_time'], name='v').chunk(azimuth_time=300, slant_range_time=300)
    
        cmlc = xr.Dataset(
            data_vars=dict(
                v=(["azimuth_time", "slant_range_time"], cmlc.data),
            ),
            coords=dict(
                azimuth_time=("azimuth_time", cstack[0].azimuth_time.coarsen(azimuth_time=w_az, boundary='pad').mean().data),
                slant_range_time=("slant_range_time", cstack[0].slant_range_time.coarsen(slant_range_time=w_rg, boundary='pad').mean().data),
            ),
            attrs=dict(description="Co-registered Multi-look Complex"),
        )
    
        rosel.io.write(cmlc, cmlc_temp_file_name)

        # generate and save GMLC (temp)
        gmlc_temp_file_name = f"{base_dir}/gmlc_temp"
        try:
            shutil.rmtree(gmlc_temp_file_name)
        except OSError as e:
            print("Error: %s - %s." % (e.filename, e.strerror))
    
#        g = rosel.shiroma.map_geo_ap(
#            c_path=cmlc_temp_file_name,
#            simulation=selection,
#            data_path_shiroma=data_path_shiroma,
#            data_path_geo_ap=gmlc_temp_file_name,
#            multilook=multilook,
#            read=True,
#            enable_dask=True,
#        )

        g = rosel.shiroma.map_geo_ap_new(
            c_path=cmlc_temp_file_name,
            data_path_shiroma=data_path_shiroma,
            data_path_geo_ap=gmlc_temp_file_name,
            multilook=multilook,
            read=True,
            enable_dask=True,
        )


        # subsample (to 50 m sampling)
        #print(g.shape)
        g_coarse = g.coarsen(x=subsampling, y=subsampling, boundary='trim').mean()
        x[:] = g_coarse.x.data
        y[:] = g_coarse.y.data
        gmlc_output[n,:,:] = xr.apply_ufunc(f, g_coarse, dask='parallelized').chunk(dict(x=50,y=50))
    

In [ ]:
# initializing the DataTree
gmlc = datatree.DataTree()

for n in tqdm(range(0,len(bursts),1)):

    cstack = ds[bursts[n]].cstack

    N = len(cstack)
    n_lines, n_samples = cstack[0].shape
    n_rg = np.floor_divide(n_samples, w_rg)
    n_az = np.floor_divide(n_lines, w_az)

    burst_path = f"{base_dir}WDIR/{bursts[n].split('/')[2]}"

    selection_path = f"{burst_path}/selection"
    global_values_path = f"{burst_path}/global_values"
    rtc_ap_path = f"{burst_path}/rtc_ap"

    #selection = rosel.io.read(selection_path)
    #selection = 0

    global_values = xr.open_dataset(global_values_path, engine="zarr")
    dim_x_global = global_values.geo_xsize.values[0]
    dim_y_global = global_values.geo_ysize.values[0]
    nx = np.floor_divide(dim_x_global, subsampling)
    ny = np.floor_divide(dim_y_global, subsampling)
    
    #nx = np.floor_divide(selection.dims['x']-1, subsampling)
    #ny = np.floor_divide(selection.dims['y']-1, subsampling)
    
    phase                   = xr.DataArray(np.zeros([N, nx, ny], dtype=np.float64),dims=["acquisition_date","x","y"])
    long_term_coherence     = xr.DataArray(np.zeros([N, nx, ny], dtype=np.float64),dims=["acquisition_date","x","y"])
    short_term_coherence    = xr.DataArray(np.zeros([N, nx, ny], dtype=np.float64),dims=["acquisition_date","x","y"])
    x = np.zeros(nx)
    y = np.zeros(ny)

    ## geocoding for each acquisition date
    ## directly from CMLC product without oversampling
    geocode_3D(cstack, burst_path, ds_cmlc[bursts[n]].phase, phase, np.angle, x, y)
    geocode_3D(cstack, burst_path, ds_cmlc[bursts[n]].long_term_coherence, long_term_coherence, np.abs, x, y)
    geocode_3D(cstack, burst_path, ds_cmlc[bursts[n]].short_term_coherence, short_term_coherence, np.abs, x, y)
    
    # substituting 0's with NaN's
    phase[1:,:,:] = phase[1:,:,:].where(phase[1:,:,:] != 0, np.NaN)
    long_term_coherence = long_term_coherence.where(long_term_coherence != 0, np.NaN)
    short_term_coherence = short_term_coherence.where(short_term_coherence != 0, np.NaN)
    
        # collecting and formatting output
    gmlc_ds = xr.Dataset(
        data_vars=dict(
            phase=(["acquisition_date", "x", "y"], np.float32(phase.data)),
            long_term_coherence=(["acquisition_date", "x", "y"], np.float16(long_term_coherence.data)),
            short_term_coherence=(["acquisition_date", "x", "y"], np.float16(short_term_coherence.data)),
        ),

#        coords=dict(
#            x=("x", selection.x[0:-1].coarsen(x=subsampling, boundary='trim').mean().data),
#            y=("y", selection.y[0:-1].coarsen(y=subsampling, boundary='trim').mean().data),
#            acquisition_date=("acquisition_date", cstack.acquisition_date.data),
#        ),

        coords=dict(
            x=("x", x),
            y=("y", y),
            acquisition_date=("acquisition_date", cstack.acquisition_date.data),
        ),

        #attrs=dict(description="Geocoded Multi-look Complex"),
        attrs = ds_cmlc[bursts[n]].attrs
    )

    gmlc_ds = gmlc_ds.transpose('acquisition_date', 'y', 'x')
    
    # saving in DataTree
    gmlc[bursts[n]] = datatree.DataTree(gmlc_ds)

In [ ]:
gmlc_file_name

In [ ]:
gmlc.to_zarr(gmlc_file_name)